# 第 1 章 発展会計: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/01-development-accounting.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
alpha = 1 / 3        # 資本分配率 α（物的資本の弾力性）
output_a = 2         # 国 A の 1 人当たり所得
capital_a = 8        # 国 A の 1 人当たり物的資本
human_capital_a = 1  # 国 A の人的資本
output_b = 0.5       # 国 B の 1 人当たり所得
capital_b = 1        # 国 B の 1 人当たり物的資本
human_capital_b = 1  # 国 B の人的資本
target = "C24"       # 所得の差を分解する国（データの C01〜C24）
reference = "C01"    # 比べる相手の国

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 2 か国の TFP と、所得の差の内訳

In [ ]:
import math

from lib.chapter01.accounting import tfp

a = tfp(output_a, capital_a, human_capital_a, alpha)
b = tfp(output_b, capital_b, human_capital_b, alpha)
gap = math.log(output_b / output_a)
capital_part = alpha * math.log(capital_b / capital_a)
human_part = (1 - alpha) * math.log(human_capital_b / human_capital_a)
tfp_part = math.log(b / a)
print(f"国 A の TFP: {a:.4f}、国 B の TFP: {b:.4f}")
print(f"所得の差（対数）: {gap:.4f}")
print(f"資本の寄与の割合: {capital_part / gap:.4f}")
print(f"人的資本の寄与の割合: {human_part / gap + 0.0:.4f}")
print(f"TFP の寄与の割合: {tfp_part / gap:.4f}")

## 24 か国のデータで、所得の差を分解する

In [ ]:
import pandas as pd

from lib.chapter01.accounting import income_gap, load_countries, variance_shares

df = load_countries()
row = pd.Series(income_gap(df, reference, alpha).loc[target])
print(row.round(3).to_string())
print(f"TFP の寄与の割合: {row['tfp'] / row['total']:.3f}")
print("所得の分散を説明する割合:")
print(variance_shares(df, alpha).round(3).to_string())

## 図: 所得の差の分解

In [ ]:
from lib.chapter01.plot import plot_gap_decomposition

plot_gap_decomposition(row)

## 図: 対数の TFP と対数の所得

In [ ]:
from lib.chapter01.plot import plot_income_and_tfp

plot_income_and_tfp(df, alpha)

## 試してみよう

- `alpha = 0.5` にすると、C24 の所得の差（−4.818）に占める TFP の割合は、
  0.596 から 0.500 に下がります。24 か国の所得の分散では、0.615 から 0.519 です。
  **α の置き方で、TFP の割合は変わります**
- `target = "C12"` にすると、所得の差は −1.988 に小さくなり、
  TFP の割合は 0.656 です（C24 は 0.596）
- `capital_b = 4`（国 B の資本を 1 から 4 に）にすると、
  国 B の TFP は 0.3150 になり、資本の寄与の割合は 0.1667、
  TFP の割合は 0.8333 です。資本の差が縮むと、所得の差の大半は TFP の差になります